## Creating Azure ML Pipelines

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connect to Azure ML Workspace

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### Creating Folder to store scripts, data and pipeline config

In [ ]:
import os
# Create a folder for the pipeline step files
experiment_folder = 'diabetes_pipeline'
os.makedirs(experiment_folder, exist_ok=True)

print(experiment_folder)

### Creating the Data Processing Script

In [ ]:
%%writefile $experiment_folder/prep_diabetes.py

# import libraries
import os
import argparse
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import mlflow

with mlflow.start_run():

    # Get parameters
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-data", type=str, dest='raw_dataset_id', help='raw dataset')
    parser.add_argument('--prepped-data', type=str, dest='prepped_data', default='prepped_data', help='Folder for results')
    args = parser.parse_args()
    save_folder = args.prepped_data
    
    # load the data (passed as an input dataset)
    print("Loading Data...")
    diabetes = run.input_datasets['raw_data'].to_pandas_dataframe()

    # Log raw row count
    row_count = (len(diabetes))
    run.log('raw_rows', row_count)

    # remove nulls
    diabetes = diabetes.dropna()

    # Normalize the numeric columns
    scaler = MinMaxScaler()
    num_cols = [
        "Pregnancies",
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI",
        "DiabetesPedigreeFunction",
        "Age"
    ]
    diabetes[num_cols] = scaler.fit_transform(diabetes[num_cols])

    # Log processed rows
    row_count = (len(diabetes))
    run.log('processed_rows', row_count)

    # Save the prepped data
    print("Saving Data...")
    os.makedirs(save_folder, exist_ok=True)
    save_path = os.path.join(save_folder,'data.csv')
    diabetes.to_csv(save_path, index=False, header=True)

### Creating the Model Training Script

In [ ]:
%%writefile $experiment_folder/train_diabetes.py

# Import libraries
from azureml.core import Run, Model
import mlflow
import argparse
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

with mlflow.start_run():
    # Get parameters
    parser = argparse.ArgumentParser()
    parser.add_argument("--training-folder", type=str, dest='training_folder', help='training data folder')
    args = parser.parse_args()
    training_folder = args.training_folder

    # load the prepared data file in the training folder
    print("Loading Data...")
    file_path = os.path.join(training_folder,'data.csv')
    diabetes = pd.read_csv(file_path)

    # Separate features and labels
    X, y = diabetes[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']].values, diabetes['Outcome'].values
    
    # Split data into training set and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    # Train adecision tree model
    print('Training a decision tree model...')
    model = DecisionTreeClassifier().fit(X_train, y_train)

    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric('Accuracy', np.float(acc))

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric('AUC', np.float(auc))

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    run.log_image(name = "ROC", plot = fig)
    plt.show()

    # Save the trained model in the outputs folder
    print("Saving model...")
    os.makedirs('outputs', exist_ok=True)
    model_file = os.path.join('outputs', 'diabetes_model.pkl')
    joblib.dump(value=model, filename=model_file)

    # Register the model
    print('Registering model...')
    Model.register(workspace=run.experiment.workspace,
                model_path = model_file,
                model_name = 'diabetes_model',
                tags={'Training context':'Pipeline'},
                properties={'AUC': np.float(auc), 'Accuracy': np.float(acc)})



### Prepare a Compute Environment for the Pipeline Steps

In [ ]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

compute_target_name = "your-compute-target-name"

# Check for existing compute target
pipeline_cluster = ComputeTarget(workspace=ws, name=compute_target_name)
print('Found existing compute target instance, use it.')

### Set Pipeline run configurations

In [ ]:
from azureml.core import Environment
from azureml.core.runconfig import RunConfiguration

# Use a curated AzureML environment (already built container)
registered_env = Environment.get(
    workspace=ws,
    name="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu"
)

# Create a new runconfig object for the pipeline
pipeline_run_config = RunConfiguration()

# Use the compute you created above. 
pipeline_run_config.target = pipeline_cluster

# Assign the environment to the run configuration
pipeline_run_config.environment = registered_env

print ("Run configuration created.")

### Create and Run a Pipeline

In [ ]:
from azureml.pipeline.core import PipelineData
from azureml.pipeline.steps import PythonScriptStep

# Get the training dataset
diabetes_ds = ws.datasets.get("diabetes dataset")

# Create a PipelineData (temporary Data Reference) for the model folder
prepped_data_folder = PipelineData("prepped_data_folder", datastore=ws.get_default_datastore())

# Step 1, Run the data prep script
prep_step = PythonScriptStep(name = "Prepare Data",
                                source_directory = experiment_folder,
                                script_name = "prep_diabetes.py",
                                arguments = ['--input-data', diabetes_ds.as_named_input('raw_data'),
                                             '--prepped-data', prepped_data_folder],
                                outputs=[prepped_data_folder],
                                compute_target = pipeline_cluster,
                                runconfig = pipeline_run_config,
                                allow_reuse = True)

# Step 2, run the training script
train_step = PythonScriptStep(name = "Train and Register Model",
                                source_directory = experiment_folder,
                                script_name = "train_diabetes.py",
                                arguments = ['--training-folder', prepped_data_folder],
                                inputs=[prepped_data_folder],
                                compute_target = pipeline_cluster,
                                runconfig = pipeline_run_config,
                                allow_reuse = True)

print("Pipeline steps defined")

In [ ]:
from azureml.core import Experiment
from azureml.pipeline.core import Pipeline

# Construct the pipeline
pipeline_steps = [prep_step, train_step]
pipeline = Pipeline(workspace=ws, steps=pipeline_steps)
print("Pipeline is built.")

# Create an experiment and run the pipeline
experiment = Experiment(workspace=ws, name = 'mslearn-diabetes-pipeline')
pipeline_run = experiment.submit(pipeline, regenerate_outputs=True)
print("Pipeline submitted for execution.")

pipeline_run.wait_for_completion(show_output=True)

### Get pipeline metrics and registered models

In [ ]:
for run in pipeline_run.get_children():
    print(run.name, ':')
    metrics = run.get_metrics()
    for metric_name in metrics:
        print('\t',metric_name, ":", metrics[metric_name])

In [ ]:
from azureml.core import Model

for model in Model.list(ws):
    print(model.name, 'version:', model.version)
    for tag_name in model.tags:
        tag = model.tags[tag_name]
        print ('\t',tag_name, ':', tag)
    for prop_name in model.properties:
        prop = model.properties[prop_name]
        print ('\t',prop_name, ':', prop)
    print('\n')